# Create Morgan Fingerprints

April 15th 2025

In this notebook we will create Morgan fingerprints for all drugs in the Tahoe 100 data set. 

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc
from datasets import load_dataset


def load_drug_metadata():
    cache_candidates = sorted(
        Path.home().glob(
            ".cache/huggingface/datasets/vevotx___tahoe-100_m/drug_metadata/0.0.0/*/tahoe-100_m-train.arrow"
        )
    )
    if cache_candidates:
        arrow_path = cache_candidates[-1]
        with pa.memory_map(str(arrow_path), "r") as source:
            try:
                drug_metadata_table = ipc.open_file(source).read_all()
            except pa.ArrowInvalid:
                source.seek(0)
                drug_metadata_table = ipc.open_stream(source).read_all()
        print(f"Loaded drug_metadata from local Hugging Face cache: {arrow_path}")
        return drug_metadata_table.to_pandas()

    print("Local drug_metadata cache not found; falling back to load_dataset(...).")
    return load_dataset("vevotx/Tahoe-100M", "drug_metadata", split="train").to_pandas()


drug_metadata = load_drug_metadata()
print(f"Loaded {len(drug_metadata)} drug rows.")

/Users/liamwilson/dl_fin/deep-learning-final-project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded drug_metadata from local Hugging Face cache: /Users/liamwilson/.cache/huggingface/datasets/vevotx___tahoe-100_m/drug_metadata/0.0.0/2dc57900b7981cfcf5e211527169a0b006546a95/tahoe-100_m-train.arrow
Loaded 379 drug rows.


In [2]:
drug_metadata

,drug,targets,moa-broad,moa-fine,human-approved,clinical-trials,gpt-notes-approval,canonical_smiles,pubchem_cid
0,Talc,None,unclear,unclear,yes,yes,Talc used in pharma and cosmetics; safety unde...,[OH-].[OH-].[O-][Si]12O[Si]3(O[Si](O1)(O[Si](O...,165411828.0
1,Bortezomib,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma and mantle cell ...,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN...,387447.0
2,Ixazomib,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma treatment.,B(C(CC(C)C)NC(=O)CNC(=O)C1=C(C=CC(=C1)Cl)Cl)(O)O,25183872.0
3,Ixazomib citrate,"PSMB1, PSMB2, PSMB5",inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma treatment as par...,B1(OC(=O)C(O1)(CC(=O)O)CC(=O)O)C(CC(C)C)NC(=O)...,56844015.0
4,Lactate (calcium),None,unclear,unclear,yes,yes,"Used in medical settings, but not specifically...",C.CC(C(=O)[O-])O.[Ca+2],168311648.0
...,...,...,...,...,...,...,...,...,...
374,Verteporfin,YAP1,inhibitor/antagonist,unclear,yes,yes,Used in photodynamic therapy for macular degen...,None,NaN
375,Quinidine (15% dihydroquinidine),KCNH2,inhibitor/antagonist,unclear,yes,yes,Approved for arrhythmias as part of quinine al...,COC1=CC2=C(C=CN=C2C=C1)[C@@H]([C@H]3C[C@@H]4CC...,441074.0
376,Canagliflozin (hemihydrate),SLC5A2,inhibitor/antagonist,Glucose transporter inhibitor,yes,yes,Approved for type 2 diabetes.,CC1=C(C=C(C=C1)[C@H]2[C@@H]([C@H]([C@@H]([C@H]...,24997615.0
377,Osimertinib (mesylate),EGFR,inhibitor/antagonist,EGFR/ERBB inhibitor,yes,yes,Approved for non-small cell lung cancer treatm...,CN1C=C(C2=CC=CC=C21)C3=NC(=NC=C3)NC4=C(C=C(C(=...,78357807.0


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

save_fingerprints = True  # Set to True to save the output CSV file; set to False to skip saving and just display the results

try:
    from rdkit.Chem import rdFingerprintGenerator

    morgan_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

    def smiles_to_morgan_bitstring(smiles):
        if pd.isna(smiles):
            return pd.NA
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is None:
            return pd.NA
        return morgan_generator.GetFingerprint(molecule).ToBitString()

except ImportError:

    def smiles_to_morgan_bitstring(smiles):
        if pd.isna(smiles):
            return pd.NA
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is None:
            return pd.NA
        return AllChem.GetMorganFingerprintAsBitVect(molecule, radius=2, nBits=2048).ToBitString()


project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "data").exists()),
    Path.cwd(),
)
morganfingerprints_output_path = project_root / "data" / "Morganfingerprints.csv"

morganfingerprints_df = drug_metadata.loc[:, ["drug", "canonical_smiles", "pubchem_cid"]].copy()
morganfingerprints_df["morgan_fingerprint"] = morganfingerprints_df["canonical_smiles"].apply(
    smiles_to_morgan_bitstring
)

missing_smiles_mask = morganfingerprints_df["canonical_smiles"].isna()
invalid_fingerprint_mask = (
    morganfingerprints_df["canonical_smiles"].notna()
    & morganfingerprints_df["morgan_fingerprint"].isna()
)
non_null_fingerprints = morganfingerprints_df["morgan_fingerprint"].dropna()

assert len(morganfingerprints_df) == len(drug_metadata) == 379
assert morganfingerprints_df["drug"].nunique() == len(morganfingerprints_df)
assert non_null_fingerprints.str.len().eq(2048).all()
assert non_null_fingerprints.str.fullmatch(r"[01]{2048}").all()
assert morganfingerprints_df.loc[missing_smiles_mask, "morgan_fingerprint"].isna().all()
assert int(missing_smiles_mask.sum()) == 2
assert int(invalid_fingerprint_mask.sum()) == 0

morganfingerprints_export_df = morganfingerprints_df.loc[
    :, ["drug", "pubchem_cid", "morgan_fingerprint"]
].copy()

if save_fingerprints:
    morganfingerprints_export_df.to_csv(morganfingerprints_output_path, index=False)

morganfingerprints_summary_df = pd.DataFrame(
    {
        "value": [
            len(morganfingerprints_export_df),
            int(morganfingerprints_export_df["morgan_fingerprint"].notna().sum()),
            int(morganfingerprints_export_df["morgan_fingerprint"].isna().sum()),
            int(missing_smiles_mask.sum()),
            int(morganfingerprints_export_df["pubchem_cid"].duplicated().sum()),
            str(morganfingerprints_output_path),
        ]
    },
    index=[
        "n_rows",
        "fingerprints_computed",
        "fingerprints_missing",
        "missing_canonical_smiles_rows",
        "duplicate_pubchem_cid_rows",
        "output_path",
    ],
)

display(morganfingerprints_summary_df)
display(morganfingerprints_export_df.head())
display(
    morganfingerprints_export_df.loc[
        morganfingerprints_export_df["drug"].isin(
            ["Bortezomib", "Trametinib", "Trametinib (DMSO_TF solvate)"]
        )
    ]
)
print(f"Saved Morgan fingerprints to {morganfingerprints_output_path}")


,value
n_rows,379
fingerprints_computed,377
fingerprints_missing,2
missing_canonical_smiles_rows,2
duplicate_pubchem_cid_rows,2
output_path,/Users/liamwilson/dl_fin/deep-learning-final-p...


,drug,pubchem_cid,morgan_fingerprint
0,Talc,165411828.0,0000000000000000000000000000000000000000000000...
1,Bortezomib,387447.0,0100000000000000000000000000000000000000000000...
2,Ixazomib,25183872.0,0100000000000000000000000000000000000000010000...
3,Ixazomib citrate,56844015.0,1100000000000000000000000000000000000000010000...
4,Lactate (calcium),168311648.0,0100000000000000000000000000000000000000000000...


,drug,pubchem_cid,morgan_fingerprint
1,Bortezomib,387447.0,0100000000000000000000000000000000000000000000...
186,Trametinib,11707110.0,0000000000000000000000000000000000000000000000...
371,Trametinib (DMSO_TF solvate),11707110.0,0000000000000000000000000000000000000000000000...


Saved Morgan fingerprints to /Users/liamwilson/dl_fin/deep-learning-final-project/data/Morganfingerprints.csv


# Make a TSNE to visualize drug data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

morganfingerprints_tsne_df = drug_metadata.loc[:, ["drug", "pubchem_cid", "moa-fine"]].merge(
    morganfingerprints_export_df,
    on=["drug", "pubchem_cid"],
    how="left",
    validate="one_to_one",
)
morganfingerprints_tsne_df["moa-fine"] = (
    morganfingerprints_tsne_df["moa-fine"].fillna("unknown").astype(str)
)

morganfingerprints_tsne_valid_df = morganfingerprints_tsne_df.loc[
    morganfingerprints_tsne_df["morgan_fingerprint"].notna()
].copy()
morganfingerprint_matrix = np.vstack(
    morganfingerprints_tsne_valid_df["morgan_fingerprint"].map(
        lambda fingerprint: np.array(list(fingerprint), dtype=np.uint8)
    )
)

morganfingerprints_tsne_pca_components = min(
    50,
    morganfingerprint_matrix.shape[0],
    morganfingerprint_matrix.shape[1],
)
morganfingerprints_tsne_perplexity = min(
    30,
    max(5, (morganfingerprint_matrix.shape[0] - 1) // 3),
)

morganfingerprints_tsne_pca = PCA(
    n_components=morganfingerprints_tsne_pca_components,
    random_state=42,
)
morganfingerprints_tsne_pca_matrix = morganfingerprints_tsne_pca.fit_transform(
    morganfingerprint_matrix
)

morganfingerprints_tsne = TSNE(
    n_components=2,
    init="pca",
    learning_rate="auto",
    perplexity=morganfingerprints_tsne_perplexity,
    random_state=42,
)
morganfingerprints_tsne_embedding = morganfingerprints_tsne.fit_transform(
    morganfingerprints_tsne_pca_matrix
)

morganfingerprints_tsne_valid_df["tsne_1"] = morganfingerprints_tsne_embedding[:, 0]
morganfingerprints_tsne_valid_df["tsne_2"] = morganfingerprints_tsne_embedding[:, 1]

morganfingerprints_tsne_summary_df = pd.DataFrame(
    {
        "value": [
            int(morganfingerprints_tsne_df.shape[0]),
            int(morganfingerprints_tsne_valid_df.shape[0]),
            int(morganfingerprints_tsne_df["morgan_fingerprint"].isna().sum()),
            int(morganfingerprints_tsne_valid_df["moa-fine"].nunique()),
            morganfingerprints_tsne_pca_components,
            morganfingerprints_tsne_perplexity,
        ]
    },
    index=[
        "n_total_drugs",
        "n_embedded_drugs",
        "n_missing_fingerprints",
        "n_unique_moa_fine",
        "pca_components",
        "tsne_perplexity",
    ],
)

display(morganfingerprints_tsne_summary_df)
display(
    morganfingerprints_tsne_valid_df["moa-fine"]
    .value_counts()
    .rename_axis("moa-fine")
    .reset_index(name="drug_count")
)

sns.set_theme(style="whitegrid", context="notebook")
morganfingerprints_tsne_moa_levels = sorted(
    morganfingerprints_tsne_valid_df["moa-fine"].unique()
)
morganfingerprints_tsne_distinct_colors = (
    list(plt.get_cmap("tab20").colors)
    + list(plt.get_cmap("tab20b").colors)
    + list(plt.get_cmap("tab20c").colors)
)
assert len(morganfingerprints_tsne_distinct_colors) >= len(morganfingerprints_tsne_moa_levels)
morganfingerprints_tsne_palette = dict(
    zip(
        morganfingerprints_tsne_moa_levels,
        morganfingerprints_tsne_distinct_colors[: len(morganfingerprints_tsne_moa_levels)],
    )
)

fig, ax = plt.subplots(figsize=(15, 11))
sns.scatterplot(
    data=morganfingerprints_tsne_valid_df,
    x="tsne_1",
    y="tsne_2",
    hue="moa-fine",
    palette=morganfingerprints_tsne_palette,
    s=60,
    alpha=0.85,
    linewidth=0,
    ax=ax,
)

ax.set_title("t-SNE of Morgan Fingerprints Colored by moa-fine")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
sns.move_legend(ax, "upper left", bbox_to_anchor=(1.02, 1), title="moa-fine", frameon=False)
fig.tight_layout()
plt.show()

print("Rendered Morgan fingerprint t-SNE colored by moa-fine.")
